# 01 — Baseline: veri yükleme + CV iskeleti

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
SEED = 42

In [12]:
# Hem lokalde hem Kaggle'da çalışsın diye veri yolunu otomatik bul
if Path("/kaggle/input").exists():
    DATA = next(Path("/kaggle/input").iterdir())   # Kaggle input klasörü
elif Path("data/raw").exists():
    DATA = Path("data/raw")                          # notebook kökten çalışırsa
else:
    DATA = Path("../data/raw")                       # notebook notebooks/ içinden çalışırsa

train = pd.read_csv(DATA / "train.csv")
test  = pd.read_csv(DATA / "test_x.csv")
y = train["career_success_score"].values

print("train:", train.shape, "| test:", test.shape)   # (10000,47) ve (10000,46) bekliyoruz
print(train.isna().sum().sort_values(ascending=False).head(10))

train: (10000, 47) | test: (10000, 46)
internship_duration_months        1657
english_exam_score                 953
open_source_contribution_count     910
github_avg_stars                   910
hr_interview_score                 780
linkedin_profile_score             668
portfolio_score                    364
cgpa                                 0
attendance_rate                      0
application_year                     0
dtype: int64


## Ortalama baseline + 5-fold CV — buraya kendim yazacağım

In [15]:
oof = np.zeros(len(train))
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

for tr_idx, val_idx in kf.split(train):
    oof[val_idx] = y[tr_idx].mean()      # bu fold'un train ortalaması

mse = ((y - oof) ** 2).mean()
print("Mean-baseline CV MSE:", mse)

Mean-baseline CV MSE: 230.64693395342437


In [17]:
import lightgbm as lgb

# 1) Modele girmeyecek kolonlar
drop_train = ["student_id", "career_success_score", "mentor_feedback_text"]
cat_cols   = ["department", "university_tier", "target_role", "hobby", "preferred_social_media_platform"]

X      = train.drop(columns=drop_train)
X_test = test.drop(columns=["student_id", "mentor_feedback_text"])

# 2) Kategorikleri 'category' tipine çevir; test, train ile AYNI kategori→kod eşlemesini kullansın
for c in cat_cols:
    X[c]      = X[c].astype("category")
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

# 3) Aynı 5 fold ile OOF + test tahmini topla (mean baseline'la BİREBİR aynı iskelet)
oof       = np.zeros(len(train))
test_pred = np.zeros(len(test))

for tr_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = lgb.LGBMRegressor(
        objective="regression",   # l2 = squared error → metriğimiz MSE ile birebir
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        verbose=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],  # val MSE 100 turda iyileşmezse dur
    )
    oof[val_idx] = model.predict(X_val)
    test_pred   += model.predict(X_test) / kf.n_splits   # 5 fold'un test tahminini ortala

# 4) [0,100]'e clip
oof       = np.clip(oof, 0, 100)
test_pred = np.clip(test_pred, 0, 100)

mse = ((y - oof) ** 2).mean()
print("LightGBM baseline CV MSE:", mse)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[403]	valid_0's l2: 86.214
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[379]	valid_0's l2: 88.7745
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[370]	valid_0's l2: 76.8774
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[385]	valid_0's l2: 79.0768
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[419]	valid_0's l2: 88.4808
LightGBM baseline CV MSE: 83.82953358994722


In [19]:
submission = pd.DataFrame({
    "student_id": test["student_id"],
    "career_success_score": test_pred,
})

sample = pd.read_csv(DATA / "sample_submission.csv")
# sample_submission yalnızca 2 satırlık FORMAT örneği → uzunluğu/id'yi test'e göre doğrula
assert list(submission.columns) == list(sample.columns), "Kolon adları/sırası uyuşmuyor!"
assert len(submission) == len(test),               "Her test satırı için bir tahmin olmalı (10000)!"
assert submission["student_id"].is_unique,         "Tekrarlayan id var!"
assert submission["student_id"].notna().all(),     "Boş id var!"
assert submission["career_success_score"].notna().all(), "Boş tahmin var!"

print(submission.head())
print("satır:", len(submission),
      "| min/max:", round(submission["career_success_score"].min(), 2),
                    round(submission["career_success_score"].max(), 2))

submission.to_csv("../submissions/sub_001_lgbm_baseline_cv83.83.csv", index=False)
print("Kaydedildi: submissions/sub_001_lgbm_baseline_cv83.83.csv")

   student_id  career_success_score
0  STU_010001             56.659755
1  STU_010002             69.709161
2  STU_010003             73.033253
3  STU_010004             96.167690
4  STU_010005             78.834424
satır: 10000 | min/max: 33.22 100.0
Kaydedildi: submissions/sub_001_lgbm_baseline_cv83.83.csv


In [20]:
def run_cv(X, y, X_test, cat_cols, params=None):
    if params is None:
        params = dict(objective="regression", n_estimators=2000, learning_rate=0.03,
                      num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                      random_state=SEED, verbose=-1)
    X, X_test = X.copy(), X_test.copy()
    for c in cat_cols:                                   # kategorikleri hizala (test=train kodlaması)
        X[c] = X[c].astype("category")
        X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

    oof, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
    for tr_idx, val_idx in kf.split(X):
        m = lgb.LGBMRegressor(**params)
        m.fit(X.iloc[tr_idx], y[tr_idx],
              eval_set=[(X.iloc[val_idx], y[val_idx])], eval_metric="l2",
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        oof[val_idx] = m.predict(X.iloc[val_idx])
        test_pred += m.predict(X_test) / kf.n_splits
    oof, test_pred = np.clip(oof, 0, 100), np.clip(test_pred, 0, 100)
    return oof, test_pred, ((y - oof) ** 2).mean()

In [21]:
def add_features(df):
    df = df.copy()
    # Dönüşüm/verimlilik oranları
    df["interview_rate"]      = df["interviews_attended"] / (df["applications_sent"] + 1)   # başvuru→mülakat dönüşümü
    df["award_per_hackathon"] = df["hackathon_awards"] / (df["hackathon_count"] + 1)        # katılım değil, başarı yoğunluğu
    df["stars_total"]         = df["github_avg_stars"].fillna(0) * df["github_repo_count"]  # toplam github etkisi

    tech = ["coding_score","problem_solving_score","data_structures_score","sql_score",
            "machine_learning_score","backend_score","frontend_score","cloud_score","devops_score"]
    df["tech_mean"] = df[tech].mean(axis=1)   # genel teknik seviye
    df["tech_max"]  = df[tech].max(axis=1)    # en güçlü olduğu alan
    df["tech_std"]  = df[tech].std(axis=1)    # uzman mı (yüksek std) yoksa dengeli mi

    soft = ["communication_score","teamwork_score","leadership_score","presentation_score"]
    df["soft_mean"] = df[soft].mean(axis=1)   # genel soft-skill bloğu

    df["experience_total"] = (df["internship_count"] + df["freelance_project_count"] +
                              df["real_client_project_count"] + df["hackathon_count"])     # toplam pratik deneyim
    return df

In [22]:
train_fe, test_fe = add_features(train), add_features(test)

drop_train = ["student_id", "career_success_score", "mentor_feedback_text"]
X      = train_fe.drop(columns=drop_train)
X_test = test_fe.drop(columns=["student_id", "mentor_feedback_text"])
cat_cols = ["department","university_tier","target_role","hobby","preferred_social_media_platform"]

oof_fe, testpred_fe, mse_fe = run_cv(X, y, X_test, cat_cols)
print("FE LightGBM CV MSE:", mse_fe, "  (baseline 83.83)")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[433]	valid_0's l2: 86.7755
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[424]	valid_0's l2: 88.2065
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[482]	valid_0's l2: 77.4073
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[496]	valid_0's l2: 79.319
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[405]	valid_0's l2: 86.5836
FE LightGBM CV MSE: 83.61333057121212   (baseline 83.83)


In [24]:
for c in cat_cols:
    X[c] = X[c].astype("category")

m_check = lgb.LGBMRegressor(objective="regression", n_estimators=400,
                              learning_rate=0.03, num_leaves=31,
                              random_state=SEED, verbose=-1)
m_check.fit(X, y)

imp = pd.Series(m_check.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(20))
print("---")
print(imp[["interview_rate","award_per_hackathon","stars_total",
           "tech_mean","tech_max","tech_std","soft_mean","experience_total"]])

technical_interview_score         786
project_quality_score             623
portfolio_score                   530
communication_score               473
target_role                       471
soft_mean                         443
cgpa                              438
hr_interview_score                435
cv_quality_score                  377
english_exam_score                369
linkedin_profile_score            345
application_year                  323
tech_mean                         303
github_avg_stars                  301
internship_duration_months        298
problem_solving_score             297
cloud_score                       276
applications_sent                 255
open_source_contribution_count    236
stars_total                       235
dtype: int32
---
interview_rate         161
award_per_hackathon     36
stars_total            235
tech_mean              303
tech_max               199
tech_std               158
soft_mean              443
experience_total       117
dtype: 

In [25]:
def add_text_features(df):
    df = df.copy()
    txt = df["mentor_feedback_text"].fillna("")
    
    df["text_len"]       = txt.str.len()                    # toplam karakter sayısı
    df["text_word_count"] = txt.str.split().str.len()       # kelime sayısı
    df["text_avg_word_len"] = df["text_len"] / (df["text_word_count"] + 1)  # ortalama kelime uzunluğu
    
    # Olumlu/olumsuz anahtar kelime sayımı (Türkçe)
    positive = ["başarılı","mükemmel","gelişmiş","güçlü","umut verici","dikkat çekici",
                "yetenekli","lider","üstün","harika","olumlu","iyi","yüksek","etkileyici"]
    negative = ["gelişmeli","eksik","zayıf","yetersiz","dikkat","sorun","çalışması gerekiyor",
                "düşük","başarısız","olumsuz","endişe","risk"]
    
    df["positive_word_count"] = txt.apply(lambda x: sum(x.lower().count(w) for w in positive))
    df["negative_word_count"] = txt.apply(lambda x: sum(x.lower().count(w) for w in negative))
    df["sentiment_ratio"]     = df["positive_word_count"] / (df["negative_word_count"] + 1)
    
    return df

In [26]:
train_fe2 = add_text_features(add_features(train))
test_fe2  = add_text_features(add_features(test))

X2      = train_fe2.drop(columns=["student_id","career_success_score","mentor_feedback_text"])
X2_test = test_fe2.drop(columns=["student_id","mentor_feedback_text"])

oof_fe2, testpred_fe2, mse_fe2 = run_cv(X2, y, X2_test, cat_cols)
print(f"Metin istatistikleri CV MSE: {mse_fe2:.4f}  (FE baseline: 83.61)")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[572]	valid_0's l2: 84.4591
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[478]	valid_0's l2: 87.4771
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[495]	valid_0's l2: 76.3708
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[433]	valid_0's l2: 77.6395
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[484]	valid_0's l2: 86.0905
Metin istatistikleri CV MSE: 82.3562  (FE baseline: 83.61)
